# Task 2 – Lesk algorithm on the MSH-WSD dataset

> *Make a random selection of 10 acronyms and 10 regular terms. Use Lesk algorithm to perform the word sense
> disambiguation of each of these 10 acronyms and 10 regular terms and use the ground truth to compute the
> average accuracy on regular terms and average accuracy on acronyms.*

MSH-WSD contains medical abstracts where an ambiguous word is marked, together with its correct meaning. We let Lesk to guess the meaning of the word in each abstract and then we grade its answer and we compare it against the solutions.

Lesk and MSH-WSD do not speak the same language:
- MSH-WSD writes the correct meaning as a **UMLS concept** (CUI), e.g. `C0009443`;
- Lesk answers with a **WordNet synset**, e.g. `cold.n.01`.


**Note on the Lesk implementation** \
Although the NLTK documentation describes `nltk.wsd.lesk` as "an implementation of the original Lesk
algorithm (1986)", inspecting the source code shows that it implements the **simplified Lesk** algorithm
(Kilgarriff & Rosenzweig, 2000): each sense is scored by the overlap between its WordNet gloss and the
**context words**, whereas the original algorithm (Lesk, 1986) compares the gloss of each sense with the
**glosses of the other words** in the context.


In [59]:
from pathlib import Path
import html
import re
import random
import nltk
import requests
from getpass import getpass
from nltk.corpus import wordnet as wn
from getpass import getpass
from collections import Counter
from nltk.wsd import lesk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)    # download WordNet (only the first time)

STOPWORDS = set(stopwords.words("english"))


This file "benchmark_mesh.txt" has one line per ambiguous word: the word, followed by the codes (CUIs) of all its possible
meanings. For example: AA	C0002520	C0001972

means that *AA* has two possible meanings. In the abstract files the correct answer is not written as a CUI
but as a label: `M1` = first CUI of the line, `M2` = second CUI, and so on. So we store the file as a
dictionary `word -> {"M1": CUI, "M2": CUI, ...}` to translate these labels later.

In [60]:
# Folder where MSHCorpus.zip was extracted
DATA_DIR = Path(r"C:\Users\flona\Documents\nlp_project\nlp_project_26\data\MSHCorpus")

# Dictionary: word -> {"M1": CUI, "M2": CUI, ...}
concept_map = {}

with open(DATA_DIR / "benchmark_mesh.txt", encoding="iso-8859-1") as f:
    for line in f:
        parts = line.strip().split("\t")   # split the line on TAB characters
        word = parts[0]                    # first element: the ambiguous word
        cuis = parts[1:]                   # remaining elements: its possible senses
        concept_map[word] = {f"M{i}": cui for i, cui in enumerate(cuis, start=1)}

print("Number of ambiguous words:", len(concept_map))
print("AA ->", concept_map["AA"])
print("Ca ->", concept_map["Ca"])

Number of ambiguous words: 203
AA -> {'M1': 'C0002520', 'M2': 'C0001972'}
Ca -> {'M1': 'C0006675', 'M2': 'C0006754', 'M3': 'C0019564', 'M4': 'C0006823'}


There is one file per word (e.g. `AA_pmids_tagged.arff`). After the line `@DATA`, each line contains one
abstract in this format:

```
<PubMed id>,"<abstract text, with the ambiguous word marked as <e>word</e>>",M1
```

We split each line into three parts: the id (before the **first** comma), the label (after the **last**
comma) and the text in between. We cannot simply split on every comma, because the text itself contains
commas. Then we clean the text: we convert HTML codes such as `&lt;` into `<` and remove the `<e>` `</e>`
tags (after remembering which word was marked).

Each abstract becomes a small record with the clean text (`context`) and the correct answer (`cui`).

In [61]:
def load_instances(word):
    """Read all the sentences (instances) of one ambiguous word."""
    path = DATA_DIR / f"{word}_pmids_tagged.arff"
    lines = path.read_text(encoding="iso-8859-1").splitlines()

    # The sentences start right after the line "@DATA"
    start = next(i for i, l in enumerate(lines) if l.startswith("@DATA")) + 1

    instances = []
    for line in lines[start:]:
        line = line.strip()
        if not line:                       # skip empty lines
            continue

        pmid, rest = line.split(",", 1)    # text before the FIRST comma = PMID
        text, label = rest.rsplit(",", 1)  # text after the LAST comma = label (M1, M2...)
        text = html.unescape(text.strip('"'))   # remove quotes, convert &lt; -> <

        # Remember which word was marked, then remove the <e> </e> tags
        marked = re.search(r"<e>(.+?)</e>", text)
        target = marked.group(1) if marked else word
        context = re.sub(r"</?e>", "", text)

        instances.append({
            "word": word,
            "pmid": pmid,
            "target": target,                          # e.g. "cements"
            "context": context,                        # clean sentence
            "label": label.strip(),                    # e.g. "M1"
            "cui": concept_map[word][label.strip()],   # correct sense, e.g. "C0002520"
        })
    return instances


# Try it on one word
aa = load_instances("AA")
print("Number of sentences for AA:", len(aa))
print("\nFirst sentence:")
print("  target :", aa[0]["target"])
print("  label  :", aa[0]["label"], "->", aa[0]["cui"])
print("  context:", aa[0]["context"][:300], "...")

Number of sentences for AA: 198

First sentence:
  target : AA
  label  : M1 -> C0002520
  context: Effect of the duration of prefeeding on amino acid digestibility of wheat distillers dried grains with solubles in broiler chicken.The objective of this study was to determine the effect of the duration of prefeeding on prececal amino acid (AA) digestibility of wheat distillers dried grains with sol ...


The dataset does not say which words are acronyms, so we use a simple rule: a word is an acronym if it has
**at least two capital letters** (`AA`, `BPD`) or **contains a digit** (`CCl4`). Multi-word names
(`B-Cell Leukemia`) are never acronyms.

Looking at the two lists by hand, the rule makes a few mistakes, so we correct them manually:
- abbreviations with only one capital letter: *Ca* (calcium), *Ala* (alanine), *Fe* (iron), *dC*
  (deoxycytidine), *pI* (isoelectric point), *Epi*, *Staph*, *Strep*;
- normal words written in capitals: *ERUPTION*, *LABOR*, *NEUROFIBROMATOSIS*, *THYMUS*, *Gamma-Interferon*;
- three words can be both a term and an abbreviation (*EGG*, *STEM*, *Orf*), so we **exclude** them.

We obtain 108 acronyms and 92 terms, close to the numbers reported in the MSH-WSD paper (106 abbreviations,
88 terms and 9 mixed cases). Then we randomly pick 10 words from each group. `random.seed(42)` makes the
selection **reproducible**: running the notebook again gives the same 20 words.

In [62]:
# Function that checks if a word is an acronym
def is_acronym(word):
    if " " in word:
        return False
    n_upper = sum(c.isupper() for c in word)
    has_digit = any(c.isdigit() for c in word)
    return n_upper >= 2 or has_digit

# Manual corrections (checked by hand)
MANUAL_ACRONYMS = {"Ca", "Ala", "Epi", "Fe", "dC", "pI", "Staph", "Strep"}
MANUAL_TERMS    = {"ERUPTION", "LABOR", "NEUROFIBROMATOSIS", "THYMUS", "Gamma-Interferon"}
EXCLUDED        = {"EGG", "STEM", "Orf"}   # can be both a term and an acronym

all_words = sorted(w for w in concept_map if w not in EXCLUDED)
acronyms = [w for w in all_words
            if (is_acronym(w) or w in MANUAL_ACRONYMS) and w not in MANUAL_TERMS]
terms    = [w for w in all_words if w not in acronyms]

print("Acronyms:", len(acronyms))
print("Terms   :", len(terms))
print("Excluded:", sorted(EXCLUDED))

# Random selection of 10 acronyms and 10 terms 
random.seed(42)
selected_acronyms = sorted(random.sample(acronyms, 10))
selected_terms    = sorted(random.sample(terms, 10))

print("\nSelected acronyms:", selected_acronyms)
print("Selected terms   :", selected_terms)

Acronyms: 108
Terms   : 92
Excluded: ['EGG', 'Orf', 'STEM']

Selected acronyms: ['ADP', 'CAD', 'CAM', 'CDA', 'Ca', 'DDD', 'DON', 'SARS', 'Staph', 'TNT']
Selected terms   : ['B-Cell Leukemia', 'Borrelia', 'Cilia', 'Follicles', 'Ganglion', 'Nurse', 'Pneumocystis', 'Respiration', 'Sodium', 'Synapsis']


Lesk does not know anything about medicine: it can only choose among the senses listed in **WordNet**, a
general-purpose English dictionary. So, before running Lesk, we check for each word whether it exists in
WordNet and which senses it has. If the correct medical meaning is not in WordNet, Lesk can never find it.

Note: WordNet uses lowercase words and underscores instead of spaces
(`B-Cell Leukemia` → `b-cell_leukemia`), and for plurals we use `wn.morphy` to get the base form
(`follicles` → `follicle`).

In [63]:
# Function to get WordNet synsets for a word
def wordnet_senses(word):
    w = word.lower().replace(" ", "_")   # "B-Cell Leukemia" -> "b-cell_leukemia"
    senses = wn.synsets(w) 
    if not senses:                       # try the base form ("follicles" -> "follicle")
        base = wn.morphy(w)              # base form of the word
        if base:
            senses = wn.synsets(base)   
    return senses

# Show the WordNet senses of the selected words
for word in selected_acronyms + selected_terms:
    senses = wordnet_senses(word)
    n_cuis = len(concept_map[word])
    print(f"\n=== {word}  ({n_cuis} senses in MSH-WSD, {len(senses)} in WordNet)")
    if not senses:
        print("   NOT IN WORDNET")
    for s in senses:
        print(f"   {s.name():25} {s.definition()}")


=== ADP  (2 senses in MSH-WSD, 2 in WordNet)
   adenosine_diphosphate.n.01 an ester of adenosine that is converted to ATP for energy storage
   automatic_data_processing.n.01 data processing by a computer

=== CAD  (2 senses in MSH-WSD, 2 in WordNet)
   cad.n.01                  someone who is morally reprehensible
   computer-aided_design.n.01 software used in art and architecture and engineering and manufacturing to assist in precision drawing

=== CAM  (2 senses in MSH-WSD, 2 in WordNet)
   cam.n.01                  a river in east central England that flows past Cambridge to join the Ouse River
   cam.n.02                  a rotating disk shaped to convert circular into linear motion

=== CDA  (2 senses in MSH-WSD, 0 in WordNet)
   NOT IN WORDNET

=== Ca  (4 senses in MSH-WSD, 2 in WordNet)
   calcium.n.01              a white metallic element that burns with a brilliant light; the fifth most abundant element in the earth's crust; an important component of most plants and animals


**Observations**
- 4 words are not in WordNet at all: *CDA*, *DDD*, *B-Cell Leukemia*, *Pneumocystis*.
- Many words have only 1 WordNet sense but 2 in MSH-WSD (e.g. *Sodium*, *TNT*), so Lesk will always give the same answer.
- Many WordNet senses are not medical: *CAD* is "someone who is morally reprehensible", *CAM* is a river, *DON* is a Spanish nobleman.

To build the translation to WordNet we need to know what each code of the dataset means, so we ask the **UMLS REST API** for the name of each concept.

In [64]:
API_KEY = getpass("Paste your UMLS API key: ")

# Function to get the name of a concept from UMLS API
def cui_name(cui):
    url = f"https://uts-ws.nlm.nih.gov/rest/content/current/CUI/{cui}"
    r = requests.get(url, params={"apiKey": API_KEY}, timeout=20)
    if r.status_code != 200:
        return None                      # concept not found (e.g. retired CUI)
    return r.json()["result"]["name"]

# Get the names of all the senses of the 20 selected words
cui_names = {}
for word in selected_acronyms + selected_terms:
    print(f"\n=== {word}")
    for label, cui in concept_map[word].items():
        cui_names[cui] = cui_name(cui)
        print(f"   {label}  {cui}  ->  {cui_names[cui]}")


=== ADP
   M1  C0001459  ->  Adenosine Diphosphate
   M2  C0004374  ->  Automatic Data Processing

=== CAD
   M1  C0011905  ->  Computer Assisted Diagnosis
   M2  C1956346  ->  Coronary Artery Disease

=== CAM
   M1  C0007578  ->  Cell Adhesion Molecules
   M2  C0178551  ->  chorioallantoic membrane

=== CDA
   M1  C0092801  ->  cladribine
   M2  C0002876  ->  Congenital dyserythropoietic anemia

=== Ca
   M1  C0006675  ->  calcium
   M2  C0006754  ->  California
   M3  C0019564  ->  Hippocampus (Brain)
   M4  C0006823  ->  Canada

=== DDD
   M1  C0026256  ->  mitotane
   M2  C0011037  ->  Dichlorodiphenyldichloroethane

=== DON
   M1  C0028652  ->  Nurse Administrator
   M2  C0012020  ->  Diazooxonorleucine

=== SARS
   M1  C1175175  ->  Severe Acute Respiratory Syndrome
   M2  C1175743  ->  Severe acute respiratory syndrome-related coronavirus

=== Staph
   M1  C0038160  ->  Staphylococcal Infections
   M2  C0038170  ->  Genus staphylococcus

=== TNT
   M1  C0077404  ->  Troponin T


Now we compare the UMLS names with the WordNet senses and, for each CUI, we write by hand
the WordNet synset(s) with the same meaning. For example *ADP* `C0001459` (Adenosine Diphosphate) →
`adenosine_diphosphate.n.01`.

Rules we followed:
- if WordNet has **no equivalent** sense, the CUI gets an empty list `[]` (e.g. *Coronary Artery Disease*):
  Lesk can never be right on those abstracts;
- each WordNet synset is assigned to **at most one CUI**. Otherwise, a word like *Follicles* (one WordNet
  sense, two MSH-WSD senses) would make Lesk "always right" without disambiguating anything.

Some choices are subjective and are listed as limitations at the end (e.g. *Follicles*, *Respiration*,
*Nurse*).

In [65]:
# mapping of MSH-WSD senses (CUIs) to WordNet synsets, created by hand
MAPPING = {
    # ---------------- acronyms ----------------
    "ADP":   {"C0001459": ["adenosine_diphosphate.n.01"],      # Adenosine Diphosphate
              "C0004374": ["automatic_data_processing.n.01"]}, # Automatic Data Processing
    "CAD":   {"C0011905": [],                                  # Computer Assisted Diagnosis (WN has only "design")
              "C1956346": []},                                 # Coronary Artery Disease
    "CAM":   {"C0007578": [],                                  # Cell Adhesion Molecules
              "C0178551": []},                                 # chorioallantoic membrane
    "CDA":   {"C0092801": [],                                  # cladribine
              "C0002876": []},                                 # Congenital dyserythropoietic anemia
    "Ca":    {"C0006675": ["calcium.n.01"],                    # calcium
              "C0006754": ["california.n.01"],                 # California
              "C0019564": [],                                  # Hippocampus
              "C0006823": []},                                 # Canada
    "DDD":   {"C0026256": [],                                  # mitotane
              "C0011037": []},                                 # Dichlorodiphenyldichloroethane
    "DON":   {"C0028652": [],                                  # Nurse Administrator
              "C0012020": []},                                 # Diazooxonorleucine
    "SARS":  {"C1175175": ["severe_acute_respiratory_syndrome.n.01"],  # the disease
              "C1175743": []},                                 # the coronavirus
    "Staph": {"C0038160": [],                                  # Staphylococcal Infections
              "C0038170": ["staphylococcus.n.01"]},            # Genus staphylococcus
    "TNT":   {"C0077404": [],                                  # Troponin T
              "C0041070": ["tnt.n.01"]},                       # Trinitrotoluene
    # ---------------- regular terms ----------------
    "B-Cell Leukemia": {"C2004493": [],                        # Leukemia, B-Cell
                        "C0023434": []},                       # Chronic Lymphocytic Leukemia
    "Borrelia":    {"C0024198": [],                            # Lyme Disease
                    "C0006033": ["borrelia.n.01"]},            # Borrelia (bacterium)
    "Cilia":       {"C0008778": ["cilium.n.01"],               # Cilia (cell projection)
                    "C0015422": ["eyelash.n.01"]},             # Eyelash
    "Follicles":   {"C0221971": [],                            # Hair follicle
                    "C0018120": ["follicle.n.01"]},            # Ovarian Follicle (closest to WN gloss)
    "Ganglion":    {"C0017067": ["ganglion.n.01"],             # Ganglia (neural)
                    "C1258666": []},                           # Myxoid cyst
    "Nurse":       {"C0028661": ["nurse.n.01", "nurse.v.03"],  # Nurses (the profession)
                    "C0006147": ["breastfeed.v.01"]},          # Breast Feeding
    "Pneumocystis": {"C0032305": [],                           # (retired CUI, name not available)
                     "C0597258": []},                          # Genus Pneumocystis
    "Respiration": {"C0035203": ["respiration.n.02", "breathing.n.01"],  # breathing
                    "C0282636": ["respiration.n.01"]},         # Cell Respiration (metabolic)
    "Sodium":      {"C0037473": ["sodium.n.01"],               # sodium (element)
                    "C0037570": []},                           # Sodium, Dietary
    "Synapsis":    {"C0039062": [],                            # Synapses (neuron junction)
                    "C0598501": ["synapsis.n.01"]},            # Chromosome Pairing
}

# Quick summary: how many senses of each word have a WordNet equivalent just to see how many can be disambiguated by Lesk
for word, senses in MAPPING.items():
    mapped = sum(1 for syns in senses.values() if syns)
    print(f"{word:17} {mapped}/{len(senses)} senses mapped to WordNet")

ADP               2/2 senses mapped to WordNet
CAD               0/2 senses mapped to WordNet
CAM               0/2 senses mapped to WordNet
CDA               0/2 senses mapped to WordNet
Ca                2/4 senses mapped to WordNet
DDD               0/2 senses mapped to WordNet
DON               0/2 senses mapped to WordNet
SARS              1/2 senses mapped to WordNet
Staph             1/2 senses mapped to WordNet
TNT               1/2 senses mapped to WordNet
B-Cell Leukemia   0/2 senses mapped to WordNet
Borrelia          1/2 senses mapped to WordNet
Cilia             2/2 senses mapped to WordNet
Follicles         1/2 senses mapped to WordNet
Ganglion          1/2 senses mapped to WordNet
Nurse             2/2 senses mapped to WordNet
Pneumocystis      0/2 senses mapped to WordNet
Respiration       2/2 senses mapped to WordNet
Sodium            1/2 senses mapped to WordNet
Synapsis          1/2 senses mapped to WordNet


Most acronyms have **0/2** senses mapped (CAD, CAM, CDA, DDD, DON), while most terms have at least one.
So we already expect Lesk to perform much worse on acronyms.

For every abstract of the 20 words:
1. Lesk picks a WordNet sense, using the whole abstract as context (lowercased and split into words);
2. we translate the correct answer with our mapping (CUI → synsets);
3. the answer is correct if Lesk's synset is in the translated list.

We compute four numbers per word and then average them over the 10 words of each group (macro average):
- **accuracy**: correct answers / all abstracts;
- **coverage**: share of abstracts whose correct meaning exists in WordNet (the best Lesk could ever do);
- **accuracy on covered senses**: accuracy computed only on those abstracts;
- **random baseline**: expected accuracy of choosing a WordNet sense at random (e.g. 1 correct sense out of
  2 → 0.5). If Lesk is not better than this, it is not really "understanding" anything.

We also look separately at the words with **at least 2 WordNet senses**, the only ones where Lesk really has
to choose.

In [ ]:
def tokenize(text):
    """Split a text into lowercase words (letters and digits only)."""
    return re.findall(r"\w+", text.lower())


def evaluate(words, mapping, acronym_list):
    """Run Lesk on every sentence of the given words and compute the metrics."""
    results = []
    for word in words:
        senses = wordnet_senses(word)
        sense_names = [s.name() for s in senses]
        instances = load_instances(word)

        n_correct = n_covered = n_correct_covered = 0
        baseline_sum = 0.0   # expected correct answers of a random choice

        for inst in instances:
            gold = mapping[word][inst["cui"]]   # WordNet translation of the correct answer

            if senses:
                answer = lesk(tokenize(inst["context"]), word.lower(), synsets=senses)
                answer = answer.name() if answer else None
            else:
                answer = None

            is_correct = answer in gold
            n_correct += is_correct
            if gold:
                n_covered += 1
                n_correct_covered += is_correct
                # random choice: probability of picking one of the correct synsets
                baseline_sum += len(set(gold) & set(sense_names)) / len(sense_names)

        results.append({
            "word": word,
            "group": "acronym" if word in acronym_list else "term",
            "n_sentences": len(instances),
            "n_wn_senses": len(senses),
            "accuracy": n_correct / len(instances),
            "coverage": n_covered / len(instances),
            "acc_covered": n_correct_covered / n_covered if n_covered else None,
            "baseline_covered": baseline_sum / n_covered if n_covered else None,
        })
    return results


def mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else float("nan")


def print_report(results):
    # ---- per word ----
    print(f"{'word':17} {'group':8} {'n':>4} {'WN':>3} {'accuracy':>9} {'coverage':>9} "
          f"{'acc_cov':>8} {'random':>7}")
    for r in results:
        f = lambda v: f"{v:.3f}" if v is not None else "-"
        print(f"{r['word']:17} {r['group']:8} {r['n_sentences']:4} {r['n_wn_senses']:3} "
              f"{r['accuracy']:9.3f} {r['coverage']:9.3f} {f(r['acc_covered']):>8} "
              f"{f(r['baseline_covered']):>7}")

    # ---- averages per group ----
    for group in ["acronym", "term"]:
        rows = [r for r in results if r["group"] == group]
        real = [r for r in rows if r["n_wn_senses"] >= 2 and r["acc_covered"] is not None]
        print(f"\n=== {group.upper()}S ({len(rows)} words)")
        print(f"  Overall accuracy            : {mean(r['accuracy'] for r in rows):.3f}")
        print(f"  WordNet coverage            : {mean(r['coverage'] for r in rows):.3f}")
        print(f"  Accuracy on covered senses  : {mean(r['acc_covered'] for r in rows):.3f}")
        print(f"  Real choice (>=2 WN senses) : {len(real)} words -> "
              f"Lesk = {mean(r['acc_covered'] for r in real):.3f}, "
              f"random = {mean(r['baseline_covered'] for r in real):.3f}")
        print(f"    words: {[r['word'] for r in real]}")


results = evaluate(selected_acronyms + selected_terms, MAPPING, selected_acronyms)
print_report(results)

word              group       n  WN  accuracy  coverage  acc_cov  random
ADP               acronym   149   2     0.718     1.000    0.718   0.500
CAD               acronym   198   2     0.000     0.000        -       -
CAM               acronym   198   2     0.000     0.000        -       -
CDA               acronym   198   0     0.000     0.000        -       -
Ca                acronym   396   2     0.258     0.500    0.515   0.500
DDD               acronym   198   0     0.000     0.000        -       -
DON               acronym   126   7     0.000     0.000        -       -
SARS              acronym   198   1     0.500     0.500    1.000   1.000
Staph             acronym   198   1     0.500     0.500    1.000   1.000
TNT               acronym   198   1     0.500     0.500    1.000   1.000
B-Cell Leukemia   term      158   0     0.000     0.000        -       -
Borrelia          term      198   1     0.500     0.500    1.000   1.000
Cilia             term      156   2     0.628     1

### Results
- Lesk is much worse on acronyms than on terms. The main reason is coverage, since WordNet knows the correct medical meaning in only 30% of the acronym abstracts.
- The accuracy on covered senses looks high but is misleading. Many words have only one WordNet sense, so Lesk has no choice and is "right" without really disambiguating.
- When Lesk really has to choose (ADP, Ca, Cilia, Nurse, Respiration), it is only a bit better than random: 0.617 vs 0.500 on acronyms and 0.534 vs 0.405 on terms.

In the first selection many words had zero or one WordNet sense, so the results mix two different problems:
WordNet **not knowing** the medical meaning, and Lesk **choosing badly**. To separate them, we draw a second
random selection (`random.seed(7)`), this time only among the words with **at least 2 WordNet senses**.
The main selection remains the random one required by the task; this is an additional experiment.

The cell below prints, for each new word, its WordNet senses and the UMLS names of its CUIs, which we need to
build the new mapping.

In [67]:
wn_acronyms = [w for w in acronyms if len(wordnet_senses(w)) >= 2]
wn_terms    = [w for w in terms    if len(wordnet_senses(w)) >= 2]
print("Acronyms with >=2 WordNet senses:", len(wn_acronyms))
print("Terms    with >=2 WordNet senses:", len(wn_terms))

random.seed(7)
ext_acronyms = sorted(random.sample(wn_acronyms, min(10, len(wn_acronyms))))
ext_terms    = sorted(random.sample(wn_terms,    min(10, len(wn_terms))))
print("\nExtension acronyms:", ext_acronyms)
print("Extension terms   :", ext_terms)

# Show WordNet senses and UMLS names, needed to build the new mapping
for word in ext_acronyms + ext_terms:
    print(f"\n=== {word}")
    print("  WordNet:")
    for s in wordnet_senses(word):
        print(f"     {s.name():30} {s.definition()[:90]}")
    print("  MSH-WSD (UMLS):")
    for label, cui in concept_map[word].items():
        print(f"     {label}  {cui}  ->  {cui_name(cui)}")

Acronyms with >=2 WordNet senses: 35
Terms    with >=2 WordNet senses: 49

Extension acronyms: ['ALS', 'ANA', 'CI', 'CLS', 'FAS', 'GAG', 'MRS', 'PVC', 'RA', 'US']
Extension terms   : ['Callus', 'Cell', 'Cement', 'Cilia', 'ERUPTION', 'Exercises', 'Medullary', 'Milk', 'Phosphorus', 'Ventricles']

=== ALS
  WordNet:
     amyotrophic_lateral_sclerosis.n.01 thickening of tissue in the motor tracts of the lateral columns and anterior horns of the 
     aluminum.n.01                  a silvery ductile metallic element found primarily in bauxite
     alabama.n.01                   a state in the southeastern United States on the Gulf of Mexico; one of the Confederate st
  MSH-WSD (UMLS):
     M1  C0003372  ->  Antilymphocyte Serum
     M2  C0002736  ->  Amyotrophic Lateral Sclerosis

=== ANA
  WordNet:
     ana.n.01                       mother of the ancient Irish gods; sometimes identified with Danu
     ana.n.02                       a collection of anecdotes about a person or place
  MSH-W

Here we write the translation by hand, then we reuse the same `evaluate`
function. Subjective choices here: *Milk* (generic milk vs human milk), *Exercises* (physiotherapy vs
physical exercise), *Cement* (dental cement also mapped to "adhesive material"). *Cilia* was drawn in both
selections, which is expected since the two draws are independent.

In [68]:
# Manual mapping for the extension selection (same rules as MAPPING)
EXT_MAPPING = {
    # ---------------- acronyms ----------------
    "ALS": {"C0003372": [],                                         # Antilymphocyte Serum
            "C0002736": ["amyotrophic_lateral_sclerosis.n.01"]},    # Amyotrophic Lateral Sclerosis
    "ANA": {"C0002463": [],                                         # American Nurses' Association
            "C0003243": []},                                        # Antinuclear Antibodies
    "CI":  {"C0022326": [],                                         # Ivory Coast
            "C0008107": []},                                        # Chile
    "CLS": {"C0265252": [],                                         # Coffin-Lowry syndrome
            "C0343084": []},                                        # Capillary Leak Syndrome
    "FAS": {"C0015683": [],                                         # Fatty-acid synthase
            "C0015923": ["fetal_alcohol_syndrome.n.01"]},           # Fetal Alcohol Syndrome
    "GAG": {"C0017346": [],                                         # GAG Gene
            "C0017973": []},                                        # Glycosaminoglycans
    "MRS": {"C0025235": [],                                         # Melkersson-Rosenthal Syndrome
            "C0024487": []},                                        # Magnetic Resonance Spectroscopy
    "PVC": {"C0151636": ["premature_ventricular_contraction.n.01"], # Premature ventricular contractions
            "C0032624": ["polyvinyl_chloride.n.01"]},               # Polyvinyl Chloride
    "RA":  {"C0002893": [],                                         # Refractory anemias
            "C0034625": ["radium.n.01"],                            # radium
            "C0003873": []},                                        # Rheumatoid Arthritis
    "US":  {"C0041703": ["united_states.n.01"],                     # United States
            "C0041618": []},                                        # Ultrasonography
    # ---------------- regular terms ----------------
    "Callus":     {"C0006767": ["callus.n.02"],                     # Bone callus
                   "C0376154": ["callosity.n.01"]},                 # Skin callus
    "Cell":       {"C0007634": ["cell.n.02"],                       # Cells (biology)
                   "C1136359": ["cellular_telephone.n.01"]},        # Cellular Phone
    "Cement":     {"C1706094": ["cement.n.04", "cement.n.03"],      # Dental Cement Material
                   "C0011343": ["cementum.n.01"]},                  # Dental Cementum
    "Cilia":      {"C0008778": ["cilium.n.01"],                     # Cilia
                   "C0015422": ["eyelash.n.01"]},                   # Eyelash
    "ERUPTION":   {"C0015230": ["eruption.n.02"],                   # Exanthema (skin rash)
                   "C1533692": ["eruption.n.06"]},                  # Tooth eruption
    "Exercises":  {"C0452240": ["exercise.n.03"],                   # Physical therapy exercises
                   "C0015259": ["exercise.n.01"]},                  # Exercise
    "Medullary":  {"C0001629": ["medullary.a.03"],                  # Adrenal Medulla
                   "C0025148": ["medullary.a.02"]},                 # Medulla Oblongata
    "Milk":       {"C0026131": ["milk.n.01"],                       # Milk (body substance)
                   "C0026140": ["milk.n.02"]},                      # Milk, Human
    "Phosphorus": {"C0080014": [],                                  # Dietary Phosphorus
                   "C0031705": ["phosphorus.n.01"]},                # phosphorus (element)
    "Ventricles": {"C0018827": ["ventricle.n.02"],                  # Heart Ventricle
                   "C0007799": ["ventricle.n.01"]},                 # Cerebral Ventricles
}

ext_results = evaluate(ext_acronyms + ext_terms, EXT_MAPPING, ext_acronyms)
print_report(ext_results)

word              group       n  WN  accuracy  coverage  acc_cov  random
ALS               acronym   198   3     0.495     0.500    0.990   0.333
ANA               acronym   198   2     0.000     0.000        -       -
CI                acronym   183   2     0.000     0.000        -       -
CLS               acronym    34   2     0.000     0.000        -       -
FAS               acronym   198   2     0.500     0.500    1.000   0.500
GAG               acronym   198   9     0.000     0.000        -       -
MRS               acronym   166   2     0.000     0.000        -       -
PVC               acronym   198   2     0.813     1.000    0.813   0.500
RA                acronym   297   3     0.000     0.333    0.000   0.333
US                acronym   198   4     0.116     0.500    0.232   0.250
Callus            term      150   5     0.567     1.000    0.567   0.200
Cell              term      198   7     0.439     1.000    0.439   0.143
Cement            term      185   8     0.049     1

### Results of the extension
1. **Two different causes of failure.**
   - Even choosing only acronyms that exist in WordNet, coverage stays at 28%: WordNet has *MRS* ("a form of address for a married woman"), *GAG* ("a humorous remark") or *CI* (curie, 101), but none of their medical meanings.
   - **Terms → an algorithm problem.** Coverage is almost complete (95%), yet Lesk is right on only 45% of
     the abstracts: even with all the information available, it chooses wrong more than half of the time.
2. Some words are worse than random: *Cement* 4.9% vs 19.2%, *Milk* 4.0% vs 14.3%, *RA* 0% vs 33.3%.
   Being worse than chance means that Lesk does not fail randomly: it **always picks the same wrong sense**.
   We check this in the next step.
3. The ranking is the same as in the first selection (terms better than acronyms), which makes the
   conclusion more reliable despite the small samples.

For the words that performed worse than random, we count how many times Lesk returns each sense.
`Counter` adds 1 to an answer every time Lesk gives it, and `most_common()` sorts them by frequency.

In [69]:
# let's see which senses Lesk picks most often (for error analysis)
for word in ["Cement", "Milk", "RA", "US"]:
    senses = wordnet_senses(word)
    answers = Counter()
    for inst in load_instances(word):
        a = lesk(tokenize(inst["context"]), word.lower(), synsets=senses)
        answers[a.name() if a else None] += 1
    print(f"\n=== {word}")
    for sense, count in answers.most_common():
        definition = wn.synset(sense).definition() if sense else ""
        print(f"   {count:4}  {sense:28} {definition[:70]}")


=== Cement
    160  cement.n.02                  a building material that is a powder made of a mixture of calcined lim
     10  cement.n.04                  any of various materials used by dentists to fill cavities in teeth
      8  cement.n.01                  concrete pavement is sometimes referred to as cement
      7  cementum.n.01                a specialized bony substance covering the root of a tooth

=== Milk
    174  milk.n.03                    a river that rises in the Rockies in northwestern Montana and flows ea
     18  milk.n.01                    a white nutritious liquid secreted by mammals and used as food by huma
      3  milk.v.01                    take milk from female mammals
      2  milk.n.02                    produced by mammary glands of female mammals for feeding their young
      1  milk.v.03                    add milk to

=== RA
    288  right_ascension.n.01         (astronomy) the equatorial coordinate specifying the angle, measured e
      8  ra.n.02

**The errors are systematic**: for *Milk* Lesk returns "a river in Montana" in 174/198 abstracts, for *RA*
"right ascension (astronomy)" in 288/297, for *US* "uracil" in 154/198, and for *Cement* "a building
material" in 160/185.

We can guess the cause of this by looking for example at the winning definition of *Milk*: "a river that rises **in the** Rockies **in** northwestern Montana **and** flows eastward **to** become…". It is long and full of **stopwords**
(*the, in, and, to, of*), which appear in every abstract. Lesk counts them as if they were meaningful, so the
definition with the most "empty words" wins. This is the same problem we saw in Task 1, where Lesk chose a
sense because of the word "to".

The error analysis suggests two simple improvements, which we compare in four versions:

- **Stopword removal**: we remove the English stopwords (NLTK list) from the context. Since Lesk counts the
  words in common, a stopword missing from the context can no longer count.
- **POS filtering**: we give Lesk only the **noun** senses (`pos='n'`), discarding verbs such as
  *milk.v.01* ("take milk from female mammals"). If a word has no noun senses (e.g. *Medullary*, an
  adjective) we keep all its senses. Side effect: for *Nurse*, breastfeeding was mapped to a verb
  (`breastfeed.v.01`), so with POS filtering Lesk can no longer get it right.


In [70]:
def candidate_senses(word, use_pos):
    """WordNet senses of the word; with use_pos only nouns (if the word has any)."""
    senses = wordnet_senses(word)
    if use_pos:
        nouns = [s for s in senses if s.pos() == "n"]
        if nouns:                 # e.g. "Medullary" has only adjectives: keep all senses
            senses = nouns
    return senses

def evaluate_variant(words, mapping, use_pos, remove_stopwords):
    """Accuracy and accuracy on covered senses of one Lesk variant, for each word."""
    scores = {}
    for word in words:
        senses = candidate_senses(word, use_pos)
        instances = load_instances(word)
        n_correct = n_covered = n_correct_covered = 0
        for inst in instances:
            gold = mapping[word][inst["cui"]]
            context = tokenize(inst["context"])
            if remove_stopwords:
                context = [t for t in context if t not in STOPWORDS]
            if senses:
                answer = lesk(context, word.lower(), synsets=senses)
                answer = answer.name() if answer else None
            else:
                answer = None
            is_correct = answer in gold
            n_correct += is_correct
            if gold:
                n_covered += 1
                n_correct_covered += is_correct
        scores[word] = (n_correct / len(instances),
                        n_correct_covered / n_covered if n_covered else None)
    return scores


VARIANTS = {
    "Lesk (base)":             dict(use_pos=False, remove_stopwords=False),
    "Lesk + POS":              dict(use_pos=True,  remove_stopwords=False),
    "Lesk + stopwords":        dict(use_pos=False, remove_stopwords=True),
    "Lesk + stopwords + POS":  dict(use_pos=True,  remove_stopwords=True),
}

SELECTIONS = {
    "Selection 1 (random)":       (selected_acronyms, selected_terms, MAPPING),
    "Selection 2 (>=2 WN senses)": (ext_acronyms, ext_terms, EXT_MAPPING),
}

all_scores = {}   # saved for the error analysis below

for sel_name, (acr, ter, mapping) in SELECTIONS.items():
    print(f"\n=== {sel_name}")
    print(f"{'variant':25} {'acr_acc':>8} {'acr_cov':>8} {'term_acc':>9} {'term_cov':>9}")
    for var_name, params in VARIANTS.items():
        scores = evaluate_variant(acr + ter, mapping, **params)
        all_scores[(sel_name, var_name)] = scores
        acr_acc  = mean(scores[w][0] for w in acr)
        acr_cov  = mean(scores[w][1] for w in acr)
        term_acc = mean(scores[w][0] for w in ter)
        term_cov = mean(scores[w][1] for w in ter)
        print(f"{var_name:25} {acr_acc:8.3f} {acr_cov:8.3f} {term_acc:9.3f} {term_cov:9.3f}")


=== Selection 1 (random)
variant                    acr_acc  acr_cov  term_acc  term_cov
Lesk (base)                  0.248    0.847     0.434     0.825
Lesk + POS                   0.248    0.847     0.436     0.829
Lesk + stopwords             0.245    0.842     0.433     0.824
Lesk + stopwords + POS       0.245    0.842     0.449     0.845

=== Selection 2 (>=2 WN senses)
variant                    acr_acc  acr_cov  term_acc  term_cov
Lesk (base)                  0.192    0.607     0.403     0.453
Lesk + POS                   0.192    0.607     0.408     0.458
Lesk + stopwords             0.256    0.881     0.425     0.475
Lesk + stopwords + POS       0.256    0.881     0.478     0.528


**Observations**
- **Selection 1**: almost no change, because most words have only one WordNet sense, so there is nothing to improve.
- **Selection 2**: removing stopwords is the most useful change, especially on acronyms (0.607 → 0.881 on covered senses).
- POS alone helps very little, because most wrong senses are nouns too (e.g. the river *Milk*).
- Stopwords + POS together is the best version on terms (0.453 → 0.528).

After removing stopwords, some abstracts may share **no word at all** with any definition. In that case
Lesk has no clue and simply returns the **first WordNet sense** (because of how ties are handled, see the note
at the top). So part of the improvement could come from this fallback rather than from real word overlap.

For each word we compute the **zero-overlap rate**: the share of abstracts in which no definition shares any
word with the context.

In [71]:
def zero_overlap_rate(word, use_pos, remove_stopwords):
    """Fraction of sentences where NO sense shares a word with the context
    (in that case Lesk simply returns the first WordNet sense)."""
    senses = candidate_senses(word, use_pos)
    if not senses:
        return None
    zero = 0
    instances = load_instances(word)
    for inst in instances:
        context = set(tokenize(inst["context"]))
        if remove_stopwords:
            context -= STOPWORDS
        best = max(len(context & set(s.definition().split())) for s in senses)
        zero += (best == 0)
    return zero / len(instances)


sel = "Selection 2 (>=2 WN senses)"
base = all_scores[(sel, "Lesk (base)")]
best = all_scores[(sel, "Lesk + stopwords + POS")]

print(f"{'word':12} {'base':>6} {'best':>6} {'change':>7} {'zero_overlap':>13}")
for word in ext_acronyms + ext_terms:
    zr = zero_overlap_rate(word, use_pos=True, remove_stopwords=True)
    zr = f"{zr:.2f}" if zr is not None else "-"
    print(f"{word:12} {base[word][0]:6.3f} {best[word][0]:6.3f} "
          f"{best[word][0] - base[word][0]:+7.3f} {zr:>13}")

word           base   best  change  zero_overlap
ALS           0.495  0.500  +0.005          0.18
ANA           0.000  0.000  +0.000          0.96
CI            0.000  0.000  +0.000          0.64
CLS           0.000  0.000  +0.000          0.41
FAS           0.500  0.500  +0.000          0.35
GAG           0.000  0.000  +0.000          0.97
MRS           0.000  0.000  +0.000          0.84
PVC           0.813  0.919  +0.106          0.05
RA            0.000  0.195  +0.195          0.52
US            0.116  0.449  +0.333          0.53
Callus        0.567  0.880  +0.313          0.20
Cell          0.439  0.389  -0.051          0.28
Cement        0.049  0.195  +0.146          0.00
Cilia         0.628  0.641  +0.013          0.74
ERUPTION      0.242  0.298  +0.056          0.55
Exercises     0.369  0.384  +0.015          0.16
Medullary     0.515  0.308  -0.207          0.39
Milk          0.040  0.455  +0.414          0.25
Phosphorus    0.500  0.500  +0.000          0.47
Ventricles    0.677 

**Observations**
- **Real improvements**: *Milk* (+41), *Callus* (+31), *Cement* (+15) have a low zero-overlap rate, so Lesk now finds useful words in common (e.g. *dental*, *teeth*).
- **Improvements by luck**: for *US* and *RA* about half of the abstracts have no overlap at all, so Lesk just returns the first WordNet sense, which happens to be correct. This is the "back-off to the most frequent sense" described by Vasilescu et al. (2004).
- **The same fallback can hurt**: for *Medullary* the first sense ("bone marrow") is wrong, so accuracy drops from 0.515 to 0.308.

Other ideas to improve Lesk algorithm: 
- **Most Frequent Sense (MFS) baseline**: always choose the most common sense. In WordNet,
  senses are listed by frequency, so we simply take the **first sense**. This tells us whether Lesk is
  better than this simple rule, and it is also what Lesk falls back on when there is zero overlap.
- **Richer signature**: the slides define the signature of a sense as its **gloss + examples**,
  but `nltk.wsd.lesk` only uses the gloss. We write our own simplified Lesk that also uses the WordNet
  **examples** and **synonyms** (lemma names) of each sense.

The target word itself is removed from the signature: it appears in every abstract, so it would give a
free point to every sense that contains it and penalise senses with a different name (e.g. *eyelash* for
*Cilia*). All methods use stopword removal and POS filtering, so they can be compared with the best variant
found above.

In [72]:
def build_signature(sense, target, use_examples):
    """Set of words describing a sense: gloss (+ examples and synonyms if use_examples)."""
    text = sense.definition()
    if use_examples:
        text += " " + " ".join(sense.examples())                        # example sentences
        text += " " + " ".join(l.replace("_", " ") for l in sense.lemma_names())  # synonyms
    words = set(tokenize(text)) - STOPWORDS
    words -= set(tokenize(target))          # remove the target word itself
    return words


def my_lesk(context, target, senses, use_examples):
    """Simplified Lesk: choose the sense whose signature shares most words with the context."""
    context = set(context)
    return max(senses, key=lambda s: len(context & build_signature(s, target, use_examples)))


# Each method receives (word, context tokens without stopwords, candidate senses) and returns a synset
METHODS = {
    "MFS (first WordNet sense)":
        lambda word, ctx, senses: senses[0],
    "NLTK Lesk (stopwords + POS)":
        lambda word, ctx, senses: lesk(ctx, word.lower(), synsets=senses),
    "Our Lesk: gloss only":
        lambda word, ctx, senses: my_lesk(ctx, word, senses, use_examples=False),
    "Our Lesk: gloss + examples + synonyms":
        lambda word, ctx, senses: my_lesk(ctx, word, senses, use_examples=True),
}


def evaluate_method(words, mapping, method):
    """Accuracy and accuracy on covered senses of one method, for each word."""
    scores = {}
    for word in words:
        senses = candidate_senses(word, use_pos=True)
        instances = load_instances(word)
        n_correct = n_covered = n_correct_covered = 0
        for inst in instances:
            gold = mapping[word][inst["cui"]]
            context = [t for t in tokenize(inst["context"]) if t not in STOPWORDS]
            answer = method(word, context, senses).name() if senses else None
            is_correct = answer in gold
            n_correct += is_correct
            if gold:
                n_covered += 1
                n_correct_covered += is_correct
        scores[word] = (n_correct / len(instances),
                        n_correct_covered / n_covered if n_covered else None)
    return scores


for sel_name, (acr, ter, mapping) in SELECTIONS.items():
    print(f"\n=== {sel_name}")
    print(f"{'method':38} {'acr_acc':>8} {'acr_cov':>8} {'term_acc':>9} {'term_cov':>9}")
    for name, method in METHODS.items():
        scores = evaluate_method(acr + ter, mapping, method)
        print(f"{name:38} {mean(scores[w][0] for w in acr):8.3f} {mean(scores[w][1] for w in acr):8.3f} "
              f"{mean(scores[w][0] for w in ter):9.3f} {mean(scores[w][1] for w in ter):9.3f}")


=== Selection 1 (random)
method                                  acr_acc  acr_cov  term_acc  term_cov
MFS (first WordNet sense)                 0.241    0.833     0.437     0.829
NLTK Lesk (stopwords + POS)               0.245    0.842     0.449     0.845
Our Lesk: gloss only                      0.247    0.847     0.449     0.845
Our Lesk: gloss + examples + synonyms     0.254    0.876     0.456     0.854

=== Selection 2 (>=2 WN senses)
method                                  acr_acc  acr_cov  term_acc  term_cov
MFS (first WordNet sense)                 0.233    0.900     0.297     0.347
NLTK Lesk (stopwords + POS)               0.256    0.881     0.478     0.528
Our Lesk: gloss only                      0.257    0.881     0.493     0.543
Our Lesk: gloss + examples + synonyms     0.268    0.918     0.508     0.558


**Observations**
- **Selection 1**: all methods give almost the same results, because most words have only one WordNet sense.
- **Lesk vs MFS on terms in Selection 2**: Lesk is clearly better than always choosing the first sense
  (0.558 vs 0.347 on covered senses). Here the context really helps.
- **Lesk vs MFS on acronyms**: MFS is as good as Lesk (0.900 vs 0.881). For acronyms such as *US*, *RA*,
  *ALS* the first WordNet sense happens to be the one used in medical texts, so the context adds little.
- Cleaning the gloss helps a bit. Our Lesk with the gloss only is slightly better than NLTK
  (0.543 vs 0.528 on terms), because we also remove punctuation and stopwords from the definition.
- **Examples and synonyms help further**, as suggested by the slides: this is the best method on both groups (acronyms 0.918, terms 0.558 on covered senses). The gain is small because WordNet examples are short and often missing.

## Summary
- **Main result**: standard Lesk reaches **24.8% on acronyms** and **43.4% on terms**.
- **Acronyms fail because of WordNet**: it is a general dictionary and does not contain most medical
  abbreviations (coverage ~30%) (See Burgun & Bodenreider (2001))
- **Terms fail because of the algorithm**: even with 95% coverage, standard Lesk is right only ~45% of the time.
- **Many errors are systematic**: long definitions full of stopwords always win (*Milk* → a river, *RA* → astronomy).
- **Improvements**: removing stopwords, filtering by POS and adding WordNet examples and synonyms to the
  signature raise the accuracy on covered senses from 0.607 to 0.918 on acronyms and from 0.453 to 0.558 on terms.
- On **terms** the improved Lesk is clearly better than always choosing the most frequent sense,  while
  on **acronyms** it is only slightly better, because most of the gain comes from falling back
  on the first WordNet sense.

**Limitations**: the CUI → WordNet mapping was done by hand (some choices are subjective, e.g. *Follicles*,
*Milk*), and 20 words per selection is a small sample.